# photonviz — gallery

A tour of what the Python bridge can draw. Every chart is a real WebGL2 plot:
wheel to zoom, drag to pan, hover for a tooltip, click a legend entry to hide a series.

In [ ]:
import numpy as np
import photonviz as pv

rng = np.random.default_rng(42)
DARK = {"theme": "dark", "height": "300px"}

## Distributions — histogram, ECDF, box/violin

In [ ]:
samples = np.concatenate([rng.normal(-1, 0.6, 4000), rng.normal(1.5, 0.9, 6000)])
pv.histogram(samples, bins=60, color="#60a5fa", plot={**DARK, "title": "Bimodal sample"})

In [ ]:
(pv.Plot(**DARK, title="ECDF — no binning choice to defend", legend=True)
   .ecdf(rng.normal(0, 1, 2000), name="normal", color="#60a5fa")
   .ecdf(rng.gumbel(0, 1, 2000), name="gumbel", color="#f472b6"))

In [ ]:
groups = [{"x": i, "values": rng.normal(i * 0.4, 0.5 + i * 0.1, 400)} for i in range(5)]
pv.box(groups, violin=True, plot={**DARK, "title": "Violin + Tukey box"})

## Fields — heatmap, contour, hexbin (each gets a colorbar)

In [ ]:
cols = rows = 80
u = np.linspace(-3, 3, cols)[None, :]
v = np.linspace(-3, 3, rows)[:, None]
z = np.sin(u * 1.5) * np.cos(v * 1.5) * np.exp(-(u**2 + v**2) / 12)
extent = {"x": [-3, 3], "y": [-3, 3]}

pv.heatmap(z.ravel(), cols, rows, extent=extent, colormap="magma",
           plot={**DARK, "title": "Heatmap · magma"})

In [ ]:
# A diverging colormap needs a domain centred on zero, or the neutral colour drifts.
pv.heatmap(z.ravel(), cols, rows, extent=extent, colormap="RdBu",
           domain=[-float(np.abs(z).max()), float(np.abs(z).max())],
           plot={**DARK, "title": "Diverging · centred on 0"})

In [ ]:
n = 25_000
hx = np.concatenate([rng.normal(-1, 0.7, n // 2), rng.normal(1.4, 0.5, n // 2)])
hy = np.concatenate([rng.normal(-0.5, 0.6, n // 2), rng.normal(1.0, 0.8, n // 2)])
pv.hexbin(hx, hy, radius=0.09, colormap="plasma", plot={**DARK, "title": "25k points, binned"})

## Fields and rasters — the matplotlib gallery types

`contourf`, `pcolormesh`, `hist2d`, `eventplot`, `streamplot` and `barbs`, with
the same names they have in matplotlib.

In [ ]:
# Filled contour bands. `lines=True` strokes the boundaries on top.
pv.contourf(z, cols, rows, extent, levels=12, lines=True, plot={**DARK, "title": "contourf"})

In [ ]:
# A colour mesh over cells that are *not* evenly spaced — linear early and
# coarse late on x, geometric on y. A heatmap cannot express this.
x_edges = np.concatenate([np.linspace(0, 5, 12), np.linspace(6, 30, 9)])
y_edges = np.geomspace(1, 200, 16)
cells = np.abs(np.sin(np.arange((len(x_edges) - 1) * (len(y_edges) - 1)) * 0.3)) * 4

pv.pcolormesh(cells, x_edges, y_edges, plot={**DARK, "title": "pcolormesh — uneven cells"})

In [ ]:
# Rectangular binning of a point cloud (use hexbin when it is dense — hexagons
# have no preferred direction, so they read density more evenly).
pv.hist2d(rng.normal(0, 1, 40_000), rng.normal(0, 1.6, 40_000), bins=[64, 44],
          colormap="magma", plot={**DARK, "title": "hist2d"})

In [ ]:
# An event raster: one row of ticks per spike train.
trains = [np.sort(rng.random(50 + i * 8) * 10) for i in range(8)]
pv.eventplot(trains, color="#a78bfa", plot={**DARK, "title": "eventplot — 8 trains"})

In [ ]:
# Streamlines of a dipole field, each coloured by its own mean speed.
m = 48
gx, gy = np.meshgrid(np.linspace(-2, 2, m), np.linspace(-2, 2, m))
d1 = np.maximum(0.05, (gx + 1) ** 2 + gy ** 2)
d2 = np.maximum(0.05, (gx - 1) ** 2 + gy ** 2)

pv.streamplot(((gx + 1) / d1 - (gx - 1) / d2).ravel(), (gy / d1 - gy / d2).ravel(),
              m, m, {"x": [-2, 2], "y": [-2, 2]}, colormap="plasma", density=1.1,
              plot={**DARK, "title": "streamplot — dipole", "equalAspect": True})

In [ ]:
# Wind barbs: speed is read off the ticks (half / full / pennant), not the length.
n = 9
bx, by = np.meshgrid(np.arange(n), np.arange(n))
speed = np.linspace(2, 65, n * n)
angle = np.linspace(0, 2 * np.pi, n * n)

pv.barbs(bx.ravel(), by.ravel(), speed * np.cos(angle), speed * np.sin(angle),
         plot={**DARK, "title": "barbs — 2 to 65 kt"})

## Custom colours

Register your brand palette once and use it by name everywhere.

In [ ]:
# `colormap` accepts inline anchor colours as well as a built-in name.
pv.heatmap(z.ravel(), cols, rows, extent=extent,
           colormap=["#0b1020", "#1d4ed8", "#22d3ee", "#fef08a"],
           plot={**DARK, "title": "Custom ramp"})

## Finance

In [ ]:
bars = 180
close = 100 + np.cumsum(rng.normal(0, 1.1, bars))
open_ = np.concatenate([[100.0], close[:-1]])
high = np.maximum(open_, close) + rng.random(bars) * 1.5
low = np.minimum(open_, close) - rng.random(bars) * 1.5
t = np.arange(bars, dtype=float)

(pv.Plot(**DARK, title="Candles + Bollinger", legend=True)
   .candlestick(t, open_, high, low, close)
   .bollinger(t, close, period=20))

In [ ]:
equity = 100 * np.cumprod(1 + rng.normal(0.0006, 0.012, 600))
(pv.Plot(**DARK, title="Equity and its underwater curve", legend=True)
   .line(np.arange(equity.size, dtype=float), equity, name="equity", color="#34d399")
   .y_axis("dd", side="right", color="#ef4444")
   .drawdown(equity, yAxis="dd"))

## Signal processing

In [ ]:
sr = 500
tt = np.arange(4096) / sr
sig = np.sin(2 * np.pi * 50 * tt) + 0.5 * np.sin(2 * np.pi * 120 * tt) + rng.normal(0, 0.6, tt.size)
pv.psd(sig, sampleRate=sr, segment=512, window="hann",
       plot={**DARK, "title": "Welch PSD — peaks at 50 and 120 Hz", "legend": True})

## Machine learning

In [ ]:
labels = (rng.random(800) < 0.4).astype(int)
scores = np.clip(np.where(labels == 1, rng.normal(0.68, 0.17, 800), rng.normal(0.36, 0.17, 800)), 0, 1)

pv.roc_curve(scores, labels, fill=True, plot={**DARK, "title": "ROC", "legend": True})

In [ ]:
# 12-D Gaussian blobs projected to 2-D, coloured by class.
D, K, per = 12, 3, 120
means = rng.normal(0, 2.4, (K, D))
data = np.concatenate([means[k] + rng.normal(0, 1, (per, D)) for k in range(K)])
cls = np.repeat(np.arange(K), per)
centred = data - data.mean(0)
comps = np.linalg.svd(centred, full_matrices=False)[2][:2]
proj = centred @ comps.T

pv.embedding(proj[:, 0], proj[:, 1], labels=cls, classNames=["cats", "dogs", "birds"], size=5,
             plot={**DARK, "title": "Embedding (PCA)", "legend": True, "pick": "xy"})

In [ ]:
cols_ = {"age": rng.normal(0, 1, 500)}
cols_["income"] = cols_["age"] * 0.8 + rng.normal(0, 0.6, 500)
cols_["debt"] = -cols_["income"] * 0.5 + rng.normal(0, 0.9, 500)
cols_["noise"] = rng.normal(0, 1, 500)

pv.corr_matrix(list(cols_.values()), names=list(cols_),
               plot={**DARK, "title": "Correlation", "equalAspect": True, "height": "340px"})

## Model architecture

`pv.model_graph(model)` takes a **PyTorch**, **Keras**, **scikit-learn** or **ONNX**
object directly. Below is a hand-written graph so the notebook has no ML dependency;
with torch installed it is one line:

```python
pv.model_graph(model, example_input=torch.randn(1, 3, 224, 224), direction="horizontal")
```

In [ ]:
block = {
    "name": "BasicBlock",
    "nodes": [
        {"id": "x",     "type": "Input",       "shape": [64, 56, 56]},
        {"id": "conv1", "type": "Conv2d",      "shape": [64, 56, 56], "params": 36864},
        {"id": "bn1",   "type": "BatchNorm2d", "shape": [64, 56, 56], "params": 128},
        {"id": "relu",  "type": "ReLU",        "shape": [64, 56, 56]},
        {"id": "conv2", "type": "Conv2d",      "shape": [64, 56, 56], "params": 36864},
        {"id": "add",   "type": "Add",         "shape": [64, 56, 56]},
    ],
    # The residual edge skips four ranks — it routes around the trunk.
    "edges": [
        {"from": "x", "to": "conv1"}, {"from": "conv1", "to": "bn1"},
        {"from": "bn1", "to": "relu"}, {"from": "relu", "to": "conv2"},
        {"from": "conv2", "to": "add"}, {"from": "x", "to": "add"},
    ],
}

pv.model_graph(block, direction="horizontal", nodeWidth=3, nodeHeight=1.4, sizeBy="params",
               plot={"theme": "dark", "hover": False, "height": "280px"})

## 3D

In [ ]:
n = 4000
tt = np.linspace(0, 12 * np.pi, n)
pv.line3d(np.cos(tt) * tt / 30, tt / 12, np.sin(tt) * tt / 30, color="#38bdf8",
          plot={"title": "Helix", "height": "360px"})

In [ ]:
# Volume raymarching on the GPU.
d = 48
g = np.linspace(-1, 1, d)
X, Y, Z = np.meshgrid(g, g, g, indexing="ij")
field = np.exp(-((X - 0.3) ** 2 + Y**2 + Z**2) * 6) + np.exp(-((X + 0.35) ** 2 + (Y - 0.2) ** 2 + Z**2) * 8)

pv.volume(field.ravel(order="F"), [d, d, d], colormap="inferno", density=0.8,
          plot={"title": "Two blobs", "autoRotate": True, "height": "380px"})

## figsize and subplots

`figsize` is matplotlib's `(width, height)` **in inches** at `dpi` (100 by
default), and `pv.subplots` returns `(figure, axes)` the same way — `axes[i, j]`,
`axes[i]` and `axes.flat` all work. The whole grid is **one** widget.

In [ ]:
fig, axes = pv.subplots(2, 2, figsize=(12, 7), sharex=True, title="One run", **DARK)

t = np.linspace(0, 12, 800)
axes[0, 0].line(t, np.exp(-t / 5) + rng.normal(0, 0.02, t.size), name="loss").title("Loss")
axes[0, 1].line(t, 1 - np.exp(-t / 4) + rng.normal(0, 0.02, t.size), name="acc").title("Accuracy")
axes[1, 0].histogram(rng.normal(0, 1, 5000), bins=40)
axes[1, 1].scatter(t, np.sin(t) * np.exp(-t / 8), size=3)
fig  # pan or zoom any pane — sharex moves the other three

In [ ]:
# Panels can span cells and pick their own kind, for a layout that is not a grid.
fig = pv.figure(figsize=(12, 7), rows=2, cols=2, **DARK)
fig.add_subplot(colspan=2).line(t, np.cumsum(rng.normal(0, 1, t.size)), name="walk")
fig.add_subplot(row=1, col=0).ecdf(rng.normal(0, 1, 3000))
fig.add_subplot(row=1, col=1, kind="polar").line(
    np.linspace(0, 2 * np.pi, 400), np.abs(np.sin(np.linspace(0, 2 * np.pi, 400) * 3)))
fig